# Accessing Spatial Data

Let's start here! If you can directly link to an image relevant to your notebook, such as [canonical logos](https://github.com/numpy/numpy/blob/main/doc/source/_static/numpylogo.svg), do so here at the top of your notebook. You can do this with MyST Markdown syntax, outlined in [this MyST guide](https://mystmd.org/guide/figures), or you edit this cell to see a demonstration. **Be sure to include `alt` text for any embedded images to make your content more accessible.**

```{image} ../thumbnails/thumbnail.png
:alt: Project Pythia logo
:width: 200px
```

---

## Overview
The purpose of this notebook is to loop through `sentinel2_data` year and .SAFE files containing satellite imagery, which will output a list of `band_data` containing the extracted spectral band information to be loaded as a raster grid and used in further notebooks.

1. Loop through year folders
2. Find .SAFE folders
3. Extract band file paths.
4. Create list containing `band_data`


## Sentinel-2 Data Structure
Sentinel-2 Data is stored in raster format with x and y coordinates. The spectral bands that will be used for indice calculations are within a `.SAFE` folder, and stored as `.jp2` file within a `granule` folder inside of `.SAFE`.

To make it easier for Python to loop through, files were previously extracted into corresponding `year` folders within the `sentinel2_data` folder. The structure of the Sentinel-2 data files is as follows, organized by year folders from 2015 to 2024.

sentinel2_data
* Year
     * S2A... .SAFE/
         * Granule/
             *   IMG_DATA/
                 * B02.jp2
                 * B03.jp2
                 * B04.jp2
                 * B08.jp2

---

## Imports

In [11]:
import sys
import os
import glob
import xarray as xr
import rioxarray as rxr
import rasterio
import pickle

## Loading Sentinel-2 (S2) Files

The first step is to set the path to the folder containing the S2 data by defining the directory and sorting by year to match the structure of the `S2` folder. 

In [2]:
data_dir = "C:/Users/sulli/DATA_Science/seagrass_project/sentinel2_data"
years = sorted(os.listdir(data_dir))

print("Years:", years)

Years: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


### Year Loop to find .SAFE files
Then using `for` loops and the python `glob` module to loop through years files to locate the number of SAFE files.

In [3]:
for year in years:
    year_path = os.path.join(data_dir, year)

    # find all safe files
    safe_folders = glob.glob(os.path.join(year_path, "*.SAFE"))
    
    print(year, "SAFE folders:", len(safe_folders))

2024 SAFE folders: 17


### Extracting the Band File Path
Scenes are snapshots of Earth taking over time. Another `for` loop will now loop through the `.SAFE` files and the `os` and `glob` modules to find the spectral bands `.jp2` files, extract to band_data, and calculate number of scenes.

| File Name | Spectral Band Color |
|---|---|
| BO4 | Red |
| B02 | Blue |
| B03 | Green |
| B08 | NIR (near infrared) |

In [5]:
band_data = []

for safe in safe_folders:
    # search inside SAFE for jp2 files
    jp2_files = glob.glob(os.path.join(safe, "**/*.jp2"), recursive=True)

    #identify bands
    bands_path = {
        "blue": [f for f in jp2_files if "_B02_10m" in f],
        "green": [f for f in jp2_files if "_B03_10m" in f],
        "red": [f for f in jp2_files if "_B04_10m" in f],
        "nir": [f for f in jp2_files if "_B08_10m" in f],
    }

    # store results in band_data list dictionary
    band_data.append({
        "year": year,
        "safe_folder": safe,
        "bands": bands_path
    })

print("Total valid Scenes:", len(band_data))

Total valid Scenes: 17


## Saving List as Pickle File

In [12]:
output_path = "C:/Users/sulli/DATA_Science/seagrass_project/processed/band_data.pkl"

#make sure folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "wb") as f:
    pickle.dump(band_data, f)

print("Saved band_data.")

Saved band_data.


---

## Summary
The output of this notebook is a saved Python pickle file containing the raster `band_data` that can be uploaded into the next notebook.

### Preprocessing Spatial Data
The following notebook will open `band_data` and preprocess by creating individual raster grids, clipping to study area and masking cloud coverage.

## Resources and references

- GeeksforGeeks. (2026, January 13). Understanding python pickling with example. https://www.geeksforgeeks.org/python/understanding-python-pickling-example/ 
- The Python Software Foundation. (2026, March 20). Glob - unix style pathname pattern expansion. Python documentation. https://docs.python.org/3/library/glob.html
- Project Pythia Notebook template